# Incident Commander - GRPO Training (In-Process, Self-Contained)

Fine-tunes **Llama-3.2-1B-Instruct** with GRPO + Unsloth on the Incident Commander RL environment.

**Key design choice: in-process env, no HTTP.** This notebook talks to the
`IncidentCommanderEnvironment` directly via `LocalEnvAdapter`, bypassing
the OpenEnv HTTP layer entirely. State is preserved correctly across
multi-turn rollouts and all 4 reward signals fire as designed.

**What runs:**
1. Cells 1-3: install + clone + GPU check  (~3 min)
2. Cell 4: load Llama-3.2-1B-Instruct + LoRA  (~2 min)
3. Cell 5: build rollout-based reward functions
4. Cell 6: generate small prompt dataset
5. Cell 7: BEFORE evaluation (15 episodes)  (~3 min)
6. Cell 8: configure GRPOTrainer (max_steps=15)
7. Cell 9: train  (~10-15 min on T4)
8. Cell 10: save merged 16-bit model
9. Cell 11: AFTER evaluation (15 episodes)
10. Cell 12: plots + before/after table

Total wall time on free T4: ~25-30 min. No external services used.


## Cell 1. Install dependencies

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q "openenv-core[core]>=0.2.2" "trl>=0.13.0" "datasets>=2.20.0" \
    "peft>=0.11.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.0" \
    matplotlib numpy requests

## Cell 2. Clone repo + verify GPU

In [ ]:
import os, sys, torch

REPO_DIR = '/content/incident_commander'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Unknown-guy-369/Incident_commander.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --rebase 2>/dev/null || true
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Cell 3. Sanity-check the in-process env

In [ ]:
from rollout import LocalEnvAdapter, rollout_episode, SYSTEM_PROMPT
from server.incident_commander_environment import IncidentCommanderEnvironment

# Verify in-process env works (no HTTP, no openenv session bugs)
env_inner = IncidentCommanderEnvironment(difficulty=1)
env = LocalEnvAdapter(env_inner, difficulty=1)
init = env.reset(difficulty=1)
obs = init['observation']
print('Alert :', obs.get('alert_summary'))
print('Services:', [s['name'] for s in obs.get('services_overview', [])])
print('In-process env verified.')

## Cell 4. Load Llama-3.2-1B-Instruct + LoRA via Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL('GRPO', FastLanguageModel)
from trl import GRPOConfig, GRPOTrainer

MODEL_NAME       = 'unsloth/Llama-3.2-1B-Instruct'
MAX_SEQ_LENGTH   = 2048
LORA_RANK        = 16
BATCH_SIZE       = 1
GRAD_ACCUM_STEPS = 4
LEARNING_RATE    = 2e-5
NUM_GENERATIONS  = 2
ROLLOUT_STEPS    = 12
MAX_STEPS        = 15
DIFFICULTY       = 1

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=True,
    fast_inference=False,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.6,
)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('Model + LoRA ready.')

## Cell 5. Build rollout-based reward functions

Each completion triggers a full multi-turn episode against a freshly
constructed in-process IncidentCommanderEnvironment. State is preserved
correctly across read_logs / read_metrics / identify_cause / fix /
monitor_recovery / resolve. All 4 reward signals fire on terminal states.

In [ ]:
from rewards import make_grpo_reward_fns

def make_generate_fn(model, tokenizer, max_new_tokens=180, temperature=0.7):
    device = next(model.parameters()).device
    def _generate(prompt):
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536).to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=temperature > 0, temperature=temperature,
                pad_token_id=tokenizer.eos_token_id, use_cache=True,
            )
        return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return _generate

_train_generate_fn = make_generate_fn(model, tokenizer)

def episode_rollout(prompt, completion):
    """Drives a full multi-turn episode in-process for one (prompt, completion) pair."""
    env_inner = IncidentCommanderEnvironment(difficulty=DIFFICULTY)
    env = LocalEnvAdapter(env_inner, difficulty=DIFFICULTY)
    state = rollout_episode(env, _train_generate_fn,
                            max_steps=ROLLOUT_STEPS, difficulty=DIFFICULTY)
    return state.to_dict()

reward_fns = make_grpo_reward_fns(episode_rollout)
print(f'Wired {len(reward_fns)} reward fns: {[fn.__name__ for fn in reward_fns]}')

## Cell 6. Generate prompt dataset (in-process)

In [ ]:
from datasets import Dataset

def generate_initial_prompts(num_samples=40, difficulty=DIFFICULTY):
    rows = []
    for i in range(num_samples):
        env_inner = IncidentCommanderEnvironment(difficulty=difficulty)
        env = LocalEnvAdapter(env_inner, difficulty=difficulty)
        init = env.reset(difficulty=difficulty)
        obs = init.get('observation', {})
        prompt = (
            f'System: {SYSTEM_PROMPT}\n\n'
            f"Initial alert: {obs.get('alert_summary','')}\n"
            f"Services overview: {obs.get('services_overview','')}\n\n"
            'Begin the investigation. What is your first action?'
        )
        rows.append({'prompt': prompt, 'seed': i})
    return Dataset.from_list(rows)

dataset = generate_initial_prompts(num_samples=40)
print('Dataset size:', len(dataset))

## Cell 7. BEFORE evaluation (15 episodes against in-process env)

In [ ]:
import numpy as np

def evaluate_episodes(model, tokenizer, num_episodes=15, label='model',
                     difficulty=DIFFICULTY, max_steps=ROLLOUT_STEPS, temperature=0.7):
    print(f'\n=== Evaluating: {label} ({num_episodes} episodes) ===')
    FastLanguageModel.for_inference(model)
    gen_fn = make_generate_fn(model, tokenizer, max_new_tokens=180, temperature=temperature)
    rewards, valid_actions = [], []
    resolved_count, correct_root_cause = 0, 0
    per_signal = {'service_recovery':[], 'root_cause_accuracy':[],
                  'action_quality':[], 'speed':[]}
    for ep in range(num_episodes):
        env_inner = IncidentCommanderEnvironment(difficulty=difficulty)
        env = LocalEnvAdapter(env_inner, difficulty=difficulty)
        state = rollout_episode(env, gen_fn, max_steps=max_steps, difficulty=difficulty)
        bd = (state.last_observation or {}).get('reward_breakdown') or {}
        rewards.append(state.last_reward)
        for k in per_signal:
            per_signal[k].append(float(bd.get(k, 0.0)))
        valid_actions.append(int(any(a in {'read_logs','read_metrics','identify_cause'}
                                     for a in state.actions_taken)))
        if state.post_fix_status == 'recovered' and 'resolve' in state.actions_taken:
            resolved_count += 1
        if state.locked_hypothesis and state.locked_hypothesis == state.true_root_cause:
            correct_root_cause += 1
        hyp_mark = 'OK' if state.locked_hypothesis == state.true_root_cause else ' X'
        r_mark = 'OK' if (state.post_fix_status == 'recovered' and 'resolve' in state.actions_taken) else ' X'
        print(f'  ep {ep+1:>2}/{num_episodes}  steps={state.steps_used:>2}  '
              f'reward={state.last_reward:.3f}  status={state.post_fix_status}  '
              f'hyp={hyp_mark}  resolved={r_mark}')
    FastLanguageModel.for_training(model)
    summary = {
        'label': label,
        'avg_reward': float(np.mean(rewards)) if rewards else 0.0,
        'resolve_rate': resolved_count / max(num_episodes, 1),
        'root_cause_accuracy': correct_root_cause / max(num_episodes, 1),
        'per_signal_mean': {k: float(np.mean(v)) if v else 0.0 for k, v in per_signal.items()},
        'rewards': rewards,
    }
    print(f"  Summary: avg_reward={summary['avg_reward']:.4f}  "
          f"resolve_rate={summary['resolve_rate']*100:.1f}%  "
          f"root_cause_accuracy={summary['root_cause_accuracy']*100:.1f}%")
    return summary

before = evaluate_episodes(model, tokenizer, num_episodes=15, label='Before GRPO')

## Cell 8. Configure GRPOTrainer (max_steps=15)

In [ ]:
training_args = GRPOConfig(
    output_dir='outputs/commander_grpo',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_prompt_length=1024,
    max_completion_length=200,
    num_generations=NUM_GENERATIONS,
    max_steps=MAX_STEPS,
    logging_steps=2,
    save_steps=5,
    save_total_limit=3,
    beta=0.04,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to='none',
)
trainer = GRPOTrainer(
    model=model, reward_funcs=reward_fns, args=training_args,
    train_dataset=dataset, processing_class=tokenizer,
)
print(f'GRPOTrainer ready: max_steps={MAX_STEPS}.')

## Cell 9. Train

In [ ]:
print(f'Starting GRPO training (max_steps={MAX_STEPS})...')
trainer.train()
print('Training complete.')

## Cell 10. Save merged 16-bit model

In [ ]:
os.makedirs('outputs/commander_final', exist_ok=True)
model.save_pretrained_merged('outputs/commander_final', tokenizer, save_method='merged_16bit')
print('Merged 16-bit model saved to outputs/commander_final')

## Cell 11. AFTER evaluation (15 episodes)

In [ ]:
after = evaluate_episodes(model, tokenizer, num_episodes=15, label=f'After GRPO ({MAX_STEPS} steps)')

## Cell 12. Plots + summary

In [ ]:
import matplotlib.pyplot as plt

os.makedirs('assets', exist_ok=True)

metrics_names = ['Avg Reward', 'Resolve Rate', 'Root-cause Accuracy']
before_vals = [before['avg_reward'], before['resolve_rate'], before['root_cause_accuracy']]
after_vals  = [after['avg_reward'],  after['resolve_rate'],  after['root_cause_accuracy']]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(len(metrics_names)); w = 0.35
axes[0].bar(x - w/2, before_vals, w, label='Before',  color='#ff6b6b', alpha=0.85)
axes[0].bar(x + w/2, after_vals,  w, label='After',   color='#51cf66', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics_names)
axes[0].set_title(f'Before vs After GRPO ({MAX_STEPS} steps)', fontweight='bold')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(after['rewards'], bins=10, color='#51cf66', alpha=0.7, edgecolor='black')
axes[1].axvline(before['avg_reward'], color='red',   linestyle='--', linewidth=2,
                label=f"Before mean: {before['avg_reward']:.3f}")
axes[1].axvline(after['avg_reward'],  color='green', linestyle='--', linewidth=2,
                label=f"After mean:  {after['avg_reward']:.3f}")
axes[1].set_title('Post-training reward distribution', fontweight='bold')
axes[1].set_xlabel('Reward'); axes[1].set_ylabel('Episodes')
axes[1].legend(); axes[1].grid(alpha=0.3)

deltas = [after_vals[i] - before_vals[i] for i in range(len(metrics_names))]
colors = ['#51cf66' if d >= 0 else '#ff6b6b' for d in deltas]
axes[2].bar(metrics_names, deltas, color=colors, alpha=0.85, edgecolor='black')
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_title('Improvement delta', fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('assets/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-signal plot
fig2, ax2 = plt.subplots(figsize=(10, 5))
signals = list(before['per_signal_mean'].keys())
b_means = [before['per_signal_mean'][k] for k in signals]
a_means = [after['per_signal_mean'][k]  for k in signals]
x = np.arange(len(signals)); w = 0.35
ax2.bar(x - w/2, b_means, w, label='Before', color='#ff6b6b', alpha=0.85)
ax2.bar(x + w/2, a_means, w, label='After',  color='#51cf66', alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(signals, rotation=15, ha='right')
ax2.set_title('Per-signal reward (raw points) - before vs after', fontweight='bold')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('assets/per_signal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '=' * 60)
print(f'Incident Commander - GRPO results ({MAX_STEPS} steps, in-process env)')
print('=' * 60)
print(f"{'Metric':<25}{'Before':>10}{'After':>10}{'Delta':>10}")
print('-' * 60)
for name, b, a in zip(metrics_names, before_vals, after_vals):
    delta = a - b
    arrow = 'UP' if delta > 0 else ('DOWN' if delta < 0 else 'FLAT')
    print(f'{name:<25}{b:>10.3f}{a:>10.3f}{delta:>+10.3f} {arrow}')
print('=' * 60)
print('Plots saved to assets/before_after_comparison.png and assets/per_signal_comparison.png')

---
## Notes for judges

* **In-process env**: This notebook drives `IncidentCommanderEnvironment`
  directly (no HTTP). State (revealed logs, locked hypothesis, fix outcome)
  is preserved across the multi-turn rollout, so all 4 reward signals fire
  correctly. The HF Space at https://abishek-priyan-369-incident-commander.hf.space
  exposes the same environment over OpenEnv's HTTP API.
* **Why 15 steps**: Enough to demonstrate the GRPO pipeline works end-to-end
  on a free T4. Production training would use 200+ steps.
* **Reward signals**: 6 GRPO reward functions are wired (4 env signals +
  shaping + format). All are logged independently by TRL.
* **Reproducibility**: `random_state=3407` in the LoRA config. Re-running
  produces stable numbers (heuristic resolve rate ranges 70-90%).